## Example 1: Decode an encoded polyline

In [ ]:
# Example 1: Decode an encoded polyline
from routingpy import utils

# The example polyline from Google's polyline-encoding documentation (precision 5).
# routingpy returns the points as (longitude, latitude) pairs.
coords = utils.decode_polyline5('_p~iF~ps|U_ulLnnqC_mqNvxq`@')
assert coords == [(-120.2, 38.5), (-120.95, 40.7), (-126.453, 43.252)], coords
print(f"✓ Decoded {len(coords)} points: {coords}")

## Example 2: Build a directions request without sending it (dry_run)

In [ ]:
# Example 2: Build a directions request without sending it
from routingpy import OSRM, get_router_by_name
from routingpy.direction import Direction

assert get_router_by_name('osrm') is OSRM

# example.invalid is a reserved name that never resolves, so nothing can reach a server.
router = OSRM(base_url='https://example.invalid')
# dry_run=True prints the URL and parameters the router would send and returns an empty result.
direction = router.directions([[8.68, 49.41], [8.69, 49.42]], profile='driving', dry_run=True)
assert isinstance(direction, Direction), type(direction)
assert direction.geometry is None and direction.distance is None and direction.duration is None, direction
print("✓ dry_run built the request without contacting a server")

## Example 3: Parse a directions response from a stand-in client

A `BaseClient` subclass returns canned OSRM JSON, so routingpy's request building and response parsing run without a routing server.

In [ ]:
# Example 3: Parse a directions response returned by a stand-in client
from routingpy import OSRM
from routingpy.client_base import BaseClient
from routingpy.direction import Direction


class CannedClient(BaseClient):
    """Records each request and answers it from `replies` instead of the network."""

    replies = {}
    requests = []

    def _request(self, url, get_params={}, post_params=None, first_request_time=None,
                 retry_counter=0, dry_run=None):
        CannedClient.requests.append((url, dict(get_params)))
        return CannedClient.replies[url.split('/')[1]]  # keyed by service: 'route', 'table', ...


# A minimal OSRM /route reply; the geometry is the polyline from Example 1.
CannedClient.replies['route'] = {
    'code': 'Ok',
    'routes': [{'geometry': '_p~iF~ps|U_ulLnnqC_mqNvxq`@', 'distance': 1234.5, 'duration': 321.9}],
}

router = OSRM(base_url='https://example.invalid', client=CannedClient)
direction = router.directions([[8.68, 49.41], [8.69, 49.42]], profile='driving')

assert CannedClient.requests[-1][0] == '/route/v1/driving/8.68,49.41;8.69,49.42', CannedClient.requests
assert isinstance(direction, Direction), type(direction)
assert direction.distance == 1234 and direction.duration == 321, direction  # routingpy keeps whole units
assert direction.geometry == [(-120.2, 38.5), (-120.95, 40.7), (-126.453, 43.252)], direction.geometry
print(f"✓ Direction: {direction.distance} m, {direction.duration} s, {len(direction.geometry)} points")

## Example 4: Parse a matrix response from the stand-in client

In [ ]:
# Example 4: Parse a matrix response returned by the stand-in client
# Uses CannedClient and router from Example 3.
from routingpy.matrix import Matrix

CannedClient.replies['table'] = {
    'code': 'Ok',
    'durations': [[0, 600.5], [610.2, 0]],
    'distances': [[0, 5000], [5100, 0]],
}

matrix = router.matrix([[8.68, 49.41], [8.69, 49.42]], profile='driving')

url, params = CannedClient.requests[-1]
assert url == '/table/v1/driving/8.68,49.41;8.69,49.42', url
assert params == {'annotations': 'duration,distance'}, params
assert isinstance(matrix, Matrix), type(matrix)
assert matrix.durations == [[0, 600.5], [610.2, 0]], matrix.durations
assert matrix.distances == [[0, 5000], [5100, 0]], matrix.distances
print(f"✓ Matrix: durations {matrix.durations}, distances {matrix.distances}")